# Biological Analysis of MOFA Factors

Biological characterization of the factors through:
- **PROGENy**: signaling pathways (ULM)
- **Hallmarks (MSigDB)**: biological processes (ORA on top 100 genes)
- **Covariates**: association with drug, moa-fine, plate, and concentration (ANOVA + FDR)

For each factor, a summary table is generated with:
- Top 10 positive and negative genes (by weight)
- Top 5 positive and negative PROGENy pathways
- Top 5 positive and negative Hallmarks ORA
- F statistic and p-value (with FDR correction) per covariate

In [ ]:
import anndata as ad
import decoupler as dc
import mofax as mfx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests




def get_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''
    
PATH_OUT = '/home/miguel-agromayor-otero/Escritorio/TFM_datos/Graficos/'

In [ ]:
# --- PATHS ---
PATH_ADATA = "/home/miguel-agromayor-otero/Escritorio/TFM_datos/mofa_adata_30f.h5ad"
PATH_OUT_GRAF = "/home/miguel-agromayor-otero/Escritorio/TFM_datos/Graficos/"
# --- Load data ---
adata = ad.read(PATH_ADATA)
weights = pd.DataFrame(
    adata.uns['mofa_weights'],
    index=adata.uns['mofa_weights_genes'],
    columns=adata.uns['mofa_weights_factors']
)

weights_t = weights.T
factor_order = sorted(weights_t.index, key=lambda x: int(x.replace("Factor", "")))
weights_t

## TOP 10 genes per Factor

In [ ]:
gene_results = []

for factor in factor_order:
    factor_weights = weights_t.loc[factor].sort_values(ascending=False)
    top_10_pos     = factor_weights.head(10)
    top_10_neg     = factor_weights.tail(10).sort_values(ascending=True)

    gene_results.append({
        "Factor":          factor,
        "Top 10 Genes (+)": ", ".join(f"{g} ({v:.2f})" for g, v in zip(top_10_pos.index, top_10_pos.values)),
        "Top 10 Genes (-)": ", ".join(f"{g} ({v:.2f})" for g, v in zip(top_10_neg.index, top_10_neg.values)),
    })

df_genes = pd.DataFrame(gene_results)
print(df_genes)

## PROGENy — Signaling Pathways (ULM). Extraction of the top 5 significant pathways, top 5 positive and top 5 negative

In [ ]:
from statsmodels.stats.multitest import multipletests
import pandas as pd
import decoupler as dc
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Calculate activities and p-values with PROGENy ---
progeny = dc.get_progeny(organism='human')
acts_progeny, pvals_progeny = dc.run_ulm(mat=weights_t, net=progeny)
print(f"PROGENy activities: {acts_progeny.shape}")

# --- 2. GLOBAL BH correction across the entire matrix (single source of truth) ---
shape = pvals_progeny.shape
padj_flat = multipletests(pvals_progeny.values.flatten(), method='fdr_bh')[1]
padj_progeny = pd.DataFrame(
    padj_flat.reshape(shape),
    index=pvals_progeny.index,
    columns=pvals_progeny.columns
)

# --- 3. Heatmap: mask non-significant values and generate asterisk matrix ---
acts_progeny_sig = acts_progeny.copy()
acts_progeny_sig[padj_progeny > 0.05] = 0
annot_matrix = padj_progeny.applymap(get_stars)

# clean up factor labels for the heatmap
acts_progeny_sig_plot = acts_progeny_sig.rename(index=lambda f: str(f).replace("Factor", "").strip())
annot_matrix_plot     = annot_matrix.rename(index=lambda f: str(f).replace("Factor", "").strip())

fig, ax = plt.subplots(figsize=(28, 12))
sns.heatmap(
    acts_progeny_sig_plot.T,
    cmap='RdBu_r', center=0,
    annot=annot_matrix_plot.T,
    annot_kws={"size": 22},
    fmt='',
    yticklabels=True,
    cbar_kws={'label': 'Pathway activity (ULM)'},
    ax=ax
)
ax.tick_params(axis='x', labelsize=32, rotation=0)
ax.tick_params(axis='y', labelsize=32, rotation=0)
cbar = ax.collections[0].colorbar
cbar.set_label('Pathway activity (ULM)', size=32)
cbar.ax.tick_params(labelsize=32)
ax.set_xlabel("Factors", fontsize=32)
ax.set_ylabel("Signaling Pathways", fontsize=32)
plt.tight_layout()
plt.savefig(PATH_OUT + "heatmap_progeny.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- 4. Top-5 summary table per factor, REUSING the global padj (not recalculated) ---
progeny_results = []
for factor in factor_order:
    scores = acts_progeny.loc[factor]
    padj_factor = padj_progeny.loc[factor]  # <- same global padj used in the heatmap
    mask = padj_factor < 0.05
    scores_sig = scores[mask]
    scores_pos = scores_sig[scores_sig > 0].sort_values(ascending=False).head(5)
    scores_neg = scores_sig[scores_sig < 0].sort_values(ascending=True).head(5)

    progeny_results.append({
        "Factor": factor,
        "PROGENy Top 5 (+)": ", ".join(f"{n} ({v:.2f})" for n, v in zip(scores_pos.index, scores_pos.values)),
        "PROGENy Top 5 (-)": ", ".join(f"{n} ({v:.2f})" for n, v in zip(scores_neg.index, scores_neg.values)),
    })

df_resumen_progeny = pd.DataFrame(progeny_results)
print(df_resumen_progeny)

In [ ]:
msigdb    = dc.get_resource('MSigDB')
hallmarks = msigdb[msigdb['collection'] == 'hallmark'].drop_duplicates(subset=['geneset', 'genesymbol'])
# Binarized matrices: 1 = gene in top 100 for that factor
mat_pos = pd.DataFrame(0, index=weights_t.index, columns=weights_t.columns)
mat_neg = pd.DataFrame(0, index=weights_t.index, columns=weights_t.columns)
for factor in weights_t.index:
    mat_pos.loc[factor, weights_t.loc[factor].nlargest(100).index]  = 1
    mat_neg.loc[factor, weights_t.loc[factor].nsmallest(100).index] = 1

acts_ora_pos, pvals_ora_pos = dc.run_ora(mat=mat_pos, net=hallmarks, source='geneset', target='genesymbol')
acts_ora_neg, pvals_ora_neg = dc.run_ora(mat=mat_neg, net=hallmarks, source='geneset', target='genesymbol')

mat_pos_sig  = pd.DataFrame(float('nan'), index=factor_order, columns=acts_ora_pos.columns)
mat_neg_sig  = pd.DataFrame(float('nan'), index=factor_order, columns=acts_ora_neg.columns)
mat_pos_padj = pd.DataFrame(float('nan'), index=factor_order, columns=acts_ora_pos.columns)
mat_neg_padj = pd.DataFrame(float('nan'), index=factor_order, columns=acts_ora_neg.columns)

ora_results = []
for factor in factor_order:
    # --- Significant positive scores ---
    scores_pos = acts_ora_pos.loc[factor]
    pvals_pos = pvals_ora_pos.loc[factor]
    padj_pos = pd.Series(
        multipletests(pvals_pos, method="fdr_bh")[1],
        index=pvals_pos.index
    )
    mask_pos = padj_pos < 0.05
    scores_sig_pos = scores_pos[mask_pos]

    # --- Significant negative scores ---
    scores_neg = acts_ora_neg.loc[factor]
    pvals_neg = pvals_ora_neg.loc[factor]
    padj_neg = pd.Series(
        multipletests(pvals_neg, method="fdr_bh")[1],
        index=pvals_neg.index
    )
    mask_neg = padj_neg < 0.05
    scores_sig_neg = scores_neg[mask_neg]

    scores_pos = scores_sig_pos.sort_values(ascending=False).head(5)
    scores_neg = scores_sig_neg.sort_values(ascending=False).head(5)

    # Matrices for heatmap (scores) and for asterisks (padj)
    mat_pos_sig.loc[factor, scores_pos.index]  = scores_pos.values
    mat_neg_sig.loc[factor, scores_neg.index]  = scores_neg.values
    mat_pos_padj.loc[factor, scores_pos.index] = padj_pos[scores_pos.index].values
    mat_neg_padj.loc[factor, scores_neg.index] = padj_neg[scores_neg.index].values

    ora_results.append({
        "Factor":                  factor,
        "Hallmarks ORA Top 5 (+)": ", ".join(f"{n} ({v:.2f})" for n, v in zip(scores_pos.index, scores_pos.values)),
        "Hallmarks ORA Top 5 (-)": ", ".join(f"{n} ({v:.2f})" for n, v in zip(scores_neg.index, scores_neg.values)),
    })

df_hallmarks_ora = pd.DataFrame(ora_results)
print(df_hallmarks_ora)

In [ ]:
# quedarme solo con hallmarks presentes en algún factor
cols = mat_pos_sig.notna().any(axis=0)
mat_plot = mat_pos_sig.loc[:, cols]
padj_plot = mat_pos_padj.loc[:, cols].reindex(index=mat_plot.index, columns=mat_plot.columns)

# limpiar etiquetas: quitar "HALLMARK_" de los hallmarks y "Factor" de los factores
mat_plot = mat_plot.rename(
    columns=lambda c: c.replace("HALLMARK_", ""),   
    index=lambda f: f.replace("Factor", "").strip()  
)
padj_plot = padj_plot.rename(
    columns=lambda c: c.replace("HALLMARK_", ""),
    index=lambda f: f.replace("Factor", "").strip()
)

# matriz de asteriscos alineada celda a celda
annot_pos = padj_plot.applymap(get_stars)

fig, ax = plt.subplots(figsize=(20, 10))
sns.heatmap(
    mat_plot.T,
    cmap='Reds',          # score ORA es siempre positivo, no divergente
    annot=annot_pos.T,
    fmt='',
    yticklabels=True,
    cbar_kws={'label': 'Enriquecimiento ORA (-log10 p)'},
    ax=ax
)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
cbar = ax.collections[0].colorbar
cbar.set_label('Enriquecimiento ORA (-log10 p)', size=16)
cbar.ax.tick_params(labelsize=12)
ax.set_title("Enriquecimiento Hallmark (extremo positivo) por Factor de MOFA", fontsize=16)
ax.set_xlabel("Factores MOFA", fontsize=16)
ax.set_ylabel("Hallmarks", fontsize=16)
plt.tight_layout()
plt.savefig(PATH_OUT + "heatmap_hallmark_pos.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# keep only hallmarks present in at least one factor
cols = mat_pos_sig.notna().any(axis=0)
mat_plot = mat_pos_sig.loc[:, cols]
padj_plot = mat_pos_padj.loc[:, cols].reindex(index=mat_plot.index, columns=mat_plot.columns)

# clean up labels: remove "HALLMARK_" from hallmarks and "Factor" from factors
mat_plot = mat_plot.rename(
    columns=lambda c: c.replace("HALLMARK_", ""),
    index=lambda f: f.replace("Factor", "").strip()
)
padj_plot = padj_plot.rename(
    columns=lambda c: c.replace("HALLMARK_", ""),
    index=lambda f: f.replace("Factor", "").strip()
)

# asterisk matrix aligned cell by cell
annot_pos = padj_plot.applymap(get_stars)

fig, ax = plt.subplots(figsize=(20, 10))
sns.heatmap(
    mat_plot.T,
    cmap='Reds',          # ORA score is always positive, non-divergent
    annot=annot_pos.T,
    fmt='',
    yticklabels=True,
    cbar_kws={'label': 'ORA enrichment (-log10 p)'},
    ax=ax
)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
cbar = ax.collections[0].colorbar
cbar.set_label('ORA enrichment (-log10 p)', size=16)
cbar.ax.tick_params(labelsize=12)
ax.set_title("Hallmark enrichment (positive extreme) by MOFA Factor", fontsize=16)
ax.set_xlabel("MOFA Factors", fontsize=16)
ax.set_ylabel("Hallmarks", fontsize=16)
plt.tight_layout()
plt.savefig(PATH_OUT + "heatmap_hallmark_pos.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
import anndata as ad
import pandas as pd
import decoupler as dc

# --- Paths ---
PATH_ADATA = "/home/miguel-agromayor-otero/Escritorio/TFM_datos/mofa_adata_30f.h5ad"
PATH_OUT_GRAF = "/home/miguel-agromayor-otero/Escritorio/TFM_datos/Graficos/"
# --- Load data ---
adata = ad.read_h5ad(PATH_ADATA)

# factors x samples (using factor scores, not weights)
factor_scores = pd.DataFrame(
    adata.X,
    index=adata.obs['drug'],
    columns=adata.uns['mofa_weights_factors']
  ).T  # factors x samples
factor_scores = factor_scores.T.groupby(level='drug').mean().T
net = adata.obs[['moa-fine', 'drug']].drop_duplicates().reset_index(drop=True)
net['weight'] = 1

acts_moa, pvals_moa = dc.run_ulm(mat=factor_scores, net=net, source='moa-fine', target='drug', weight='weight')

from statsmodels.stats.multitest import multipletests
import pandas as pd

# flatten, correct, and reshape back into a matrix
shape = pvals_moa.shape
padj = multipletests(pvals_moa.values.flatten(), method='fdr_bh')[1]
padj_moa = pd.DataFrame(padj.reshape(shape),
                            index=pvals_moa.index,
                            columns=pvals_moa.columns)

# use padj instead of pvals for the asterisks and masking

acts_moa_sig = acts_moa.copy()
acts_moa_sig[padj_moa > 0.05] = 0


In [ ]:
drug_counts = adata.obs.groupby('moa-fine')['drug'].nunique()

fig, ax = plt.subplots(figsize=(56, 24))
acts_moa_sig.index = acts_moa_sig.index.str.replace('Factor', '')

sns.heatmap(acts_moa_sig.T,
            cmap='RdBu_r',
            center=0,
            linewidths=0.5,
            yticklabels=True,
            xticklabels=True,
            )

new_ylabels = [f"{label.get_text()} (n={drug_counts.get(label.get_text(), 0)})" 
               for label in ax.get_yticklabels()]
ax.set_yticklabels(new_ylabels, fontsize=60)
cbar = ax.collections[0].colorbar
cbar.set_label('MoA enrichment score (ULM)', size=60)
cbar.ax.tick_params(labelsize=60)
ax.tick_params(axis='x', labelsize=54)
plt.xlabel('Factors', size=60)
plt.ylabel('Mechanisms of Action', size=60)
plt.tight_layout()
plt.savefig(PATH_OUT_GRAF + 'heatmap_moas.pdf', format="pdf", bbox_inches="tight")
plt.show()

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
MOFA_MODEL = "/home/miguel-agromayor-otero/Escritorio/TFM_datos/modelo_mofa_30factors.hdf5"
model = mfx.mofa_model(MOFA_MODEL)
# sum of R² across factors within each (View, Group) → then average across groups
r2_total = (model.get_r2().groupby(["View", "Group"])["R2"].sum()   # total variance per view-group
              .groupby("View").mean()                    # average across the 14 groups
              .reset_index()
              .sort_values("R2", ascending=False))

plt.figure(figsize=(26,10))
sns.barplot(data=r2_total, x="View", y="R2",
            order=r2_total["View"], color="#4C72B0")
plt.xlabel("Cell lines", fontsize=32)
plt.ylabel("% Total variance explained", fontsize=32)
plt.xticks(rotation=90, fontsize=32)
plt.yticks(fontsize=32)
plt.tight_layout()
plt.savefig("/home/miguel-agromayor-otero/Escritorio/TFM_datos//Graficos/varianza_explicada.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
PATH_BATCH = '/home/miguel-agromayor-otero/Descargas/drug_clustering/TFM_mao/resultados/technical_batch/'
# --- Load ANOVAs with FDR correction ---
dfs_cov = {}
for cov in ['concentration', 'drug', 'moa_fine', 'plate']:
    df = pd.read_excel(PATH_BATCH + f'anova_{cov}.xlsx')
    _, pvals_fdr, _, _ = multipletests(df['P_Value'], method='fdr_bh')
    df[f'FDR_{cov}'] = pvals_fdr
    df = df.rename(columns={
        'F_Stat':   f'F_stat_{cov}',
        'P_Value':  f'pval_{cov}',
        'P_adj': f'pval_adj_{cov}',
        'Verdict': f'sig_{cov}'
    })
    dfs_cov[cov] = df

df_covariates = (dfs_cov['concentration']
    .merge(dfs_cov['drug'],     on='Factor')
    .merge(dfs_cov['moa_fine'], on='Factor')
    .merge(dfs_cov['plate'],    on='Factor')
)

# --- Final merge ---
df_final = (df_genes
    .merge(df_resumen_progeny, on='Factor')
    .merge(df_hallmarks_ora,   on='Factor')
    .merge(df_covariates,      on='Factor')
)

print(f"Final table: {df_final.shape[0]} factors x {df_final.shape[1]} columns")

# --- Save to Excel with themed tabs ---
out = PATH_OUT + 'resumen_factores_completo.xlsx'

cols_genes    = ['Factor'] + [c for c in df_final.columns if 'Genes' in c]
cols_progeny  = ['Factor'] + [c for c in df_final.columns if 'PROGENy' in c]
cols_hallmark = ['Factor'] + [c for c in df_final.columns if 'Hallmark' in c]
cols_cov      = ['Factor'] + [c for c in df_final.columns if any(x in c for x in ['F_', 'pval_', 'FDR_', 'sig_'])]

with pd.ExcelWriter(out, engine='openpyxl') as writer:
    df_final.to_excel(writer,               sheet_name='Resumen_completo', index=False)
    df_final[cols_genes].to_excel(writer,   sheet_name='Top_Genes',        index=False)
    df_final[cols_progeny].to_excel(writer, sheet_name='PROGENy',          index=False)
    df_final[cols_hallmark].to_excel(writer,sheet_name='Hallmarks',        index=False)
    df_final[cols_cov].to_excel(writer,     sheet_name='Covariates',       index=False)

print(f"Saved to {out}")